# Fine-Tuning Transformers with Hugging Face and PyTorch

**Two practical workflows**

1. Fine-tune pre-trained BERT for **5-class Yelp review classification** using a PyTorch training loop.
2. Fine-tune BERT for **Masked Language Modeling (MLM)** on IMDB reviews using Hugging Face `Trainer`.

Each cell includes a short explanation of *what it does* and *why it matters*. Run the notebook from top to bottom.

> **Important clarification:** The supplied notes call the second workflow “SFT Trainer.” In the Hugging Face ecosystem, TRL's `SFTTrainer` is primarily designed for supervised fine-tuning of generative/causal language models. For BERT's masked-token objective, the appropriate standard workflow is `Trainer` + `DataCollatorForLanguageModeling`. This notebook uses the technically appropriate MLM workflow while explaining the distinction.

## 0. Learning goals

By the end, you should understand:

- What Hugging Face provides: models, tokenizers, datasets, and training utilities.
- How text becomes `input_ids` and `attention_mask`.
- Why padding and truncation are needed.
- How a classification head turns BERT representations into five class scores.
- How loss, backpropagation, and AdamW update model parameters.
- How MLM masks tokens and trains BERT to reconstruct them.
- How to use a fill-mask pipeline after training.

**Core distinction:** Yelp classification predicts a review label. MLM predicts masked tokens. They are different learning tasks.

## 1. Install and import libraries

**Explanation:** `transformers` provides pretrained Transformer models and tokenizers; `datasets` loads datasets; PyTorch runs tensor operations and the manual training loop. `accelerate` supports Hugging Face training utilities.

If a package was just installed in an already-running notebook, restart the kernel if imports fail.

In [ ]:
%pip install -q transformers datasets accelerate evaluate scikit-learn

In [ ]:
import os
import random
import numpy as np
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForMaskedLM,
    get_linear_schedule_with_warmup,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    pipeline,
)
from datasets import load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


## 2. Practical A — BERT review classification

### 2.1 Load Yelp Review Full

**Explanation:** `load_dataset()` downloads the dataset and returns train/test splits. Yelp Review Full has five sentiment/rating categories labelled `0` through `4` (representing one through five stars).

The full dataset is fairly large. To make experimentation possible on a modest computer, the next cell lets you use a smaller subset. Set the limits to `None` to use the full split.

In [ ]:
raw_yelp = load_dataset("Yelp/yelp_review_full")
TRAIN_LIMIT = 8000   # Change to None to use the full training split
TEST_LIMIT = 2000    # Change to None to use the full test split

train_yelp = raw_yelp["train"]
test_yelp = raw_yelp["test"]

if TRAIN_LIMIT is not None:
    train_yelp = train_yelp.shuffle(seed=SEED).select(range(min(TRAIN_LIMIT, len(train_yelp))))
if TEST_LIMIT is not None:
    test_yelp = test_yelp.shuffle(seed=SEED).select(range(min(TEST_LIMIT, len(test_yelp))))

print(train_yelp)
print(test_yelp)
print("\nExample review:", train_yelp[0]["text"])
print("Example label:", train_yelp[0]["label"])

README.md:   0%|          | 0.00/6.72k [00:00<?, ?B/s]

yelp_review_full/train-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  299MB            

yelp_review_full/train-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

yelp_review_full/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 23.5MB            

yelp_review_full/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/650000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'text'],
    num_rows: 8000
})
Dataset({
    features: ['label', 'text'],
    num_rows: 2000
})

Example review: I stalk this truck.  I've been to industrial parks where I pretend to be a tech worker standing in line, strip mall parking lots, and of course the farmer's market.  The bowls are so so absolutely divine.  The owner is super friendly and he makes each bowl by hand with an incredible amount of pride.  You gotta eat here guys!!!
Example label: 4


### 2.2 Load the BERT tokenizer

**Explanation:** A Transformer cannot directly process ordinary text. The tokenizer splits text into tokens and maps each token to an integer ID. `bert-base-uncased` lowercases input before tokenization.

The tokenizer and model must come from compatible checkpoints so that token IDs have the meanings the model expects.

In [ ]:
MODEL_CHECKPOINT = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

example = "This restaurant has amazing food!"
encoded_example = tokenizer(example)
print(encoded_example)
print("Tokens:", tokenizer.convert_ids_to_tokens(encoded_example["input_ids"]))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

{'input_ids': [101, 2023, 4825, 2038, 6429, 2833, 999, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}
Tokens: ['[CLS]', 'this', 'restaurant', 'has', 'amazing', 'food', '!', '[SEP]']


### 2.3 Understand tokenization, padding, and truncation

**Explanation:**

- `input_ids`: integer IDs representing tokens.
- `attention_mask`: `1` for actual tokens and `0` for padding positions.
- `token_type_ids`: segment identifiers used by BERT for paired inputs; for a single review, they are usually all zero.
- `padding="max_length"` makes examples the same length.
- `truncation=True` cuts reviews longer than the chosen maximum length.

A fixed maximum length makes batches easy to stack into rectangular tensors. It also controls memory use.

In [ ]:
MAX_LENGTH = 128

def tokenize_yelp(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_train = train_yelp.map(tokenize_yelp, batched=True)
tokenized_test = test_yelp.map(tokenize_yelp, batched=True)

print(tokenized_train[0].keys())
print("First input IDs:", tokenized_train[0]["input_ids"][:20])
print("First attention mask:", tokenized_train[0]["attention_mask"][:20])

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

dict_keys(['label', 'text', 'input_ids', 'token_type_ids', 'attention_mask'])
First input IDs: [101, 1045, 23899, 2023, 4744, 1012, 1045, 1005, 2310, 2042, 2000, 3919, 6328, 2073, 1045, 9811, 2000, 2022, 1037, 6627]
First attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


### 2.4 Prepare PyTorch-ready datasets

**Explanation:** The model needs numerical tensors, not the original review strings. We remove the raw `text` column, rename `label` to `labels` (the name expected by many Transformers model classes), and set the output format to PyTorch tensors.

The model's forward pass can then calculate the classification loss when `labels` are supplied.

In [ ]:
columns_to_remove = ["text"]
tokenized_train = tokenized_train.remove_columns(columns_to_remove)
tokenized_test = tokenized_test.remove_columns(columns_to_remove)

tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_test = tokenized_test.rename_column("label", "labels")

# Keep only fields produced by the tokenizer and the target labels.
tokenized_train.set_format("torch")
tokenized_test.set_format("torch")

print(tokenized_train[0])

{'labels': tensor(4), 'input_ids': tensor([  101,  1045, 23899,  2023,  4744,  1012,  1045,  1005,  2310,  2042,
         2000,  3919,  6328,  2073,  1045,  9811,  2000,  2022,  1037,  6627,
         7309,  3061,  1999,  2240,  1010,  6167,  6670,  5581,  7167,  1010,
         1998,  1997,  2607,  1996,  7500,  1005,  1055,  3006,  1012,  1996,
        15220,  2024,  2061,  2061,  7078,  7746,  1012,  1996,  3954,  2003,
         3565,  5379,  1998,  2002,  3084,  2169,  4605,  2011,  2192,  2007,
         2019,  9788,  3815,  1997,  6620,  1012,  2017, 10657,  4521,  2182,
         4364,   999,   999,   999,   102,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0, 

### 2.5 Create DataLoaders

**Explanation:** A `DataLoader` groups individual examples into mini-batches. With a batch size of 8, BERT processes eight reviews in one forward pass.

Smaller batches use less GPU/CPU memory. Larger batches can make training faster but need more memory.

In [ ]:
BATCH_SIZE = 8

train_loader = DataLoader(tokenized_train, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(tokenized_test, batch_size=BATCH_SIZE, shuffle=False)

batch = next(iter(train_loader))
for key, value in batch.items():
    print(key, value.shape, value.dtype)

labels torch.Size([8]) torch.int64
input_ids torch.Size([8, 128]) torch.int64
token_type_ids torch.Size([8, 128]) torch.int64
attention_mask torch.Size([8, 128]) torch.int64


### 2.6 Load BERT with a classification head

**Explanation:** `AutoModelForSequenceClassification` loads the pretrained BERT encoder and attaches a task-specific classification head. `num_labels=5` means the output has five class scores—one for each Yelp rating class.

The pretrained encoder already contains language knowledge. The newly initialized classification head must learn how to map that knowledge to Yelp's five labels.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=5,
).to(device)

print(model.config.num_labels)

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


5


### 2.7 Inspect logits and loss

**Explanation:** `logits` are the model's raw class scores, with shape `[batch_size, 5]`. They are not probabilities yet. The highest-scoring class is the predicted label.

When we pass `labels`, the model also returns a classification loss. Cross-entropy loss compares the predicted scores with the correct class IDs.

In [ ]:
batch = {key: value.to(device) for key, value in batch.items()}
model.eval()
with torch.no_grad():
    outputs = model(**batch)

print("Logits shape:", outputs.logits.shape)
print("First example logits:", outputs.logits[0])
print("Predicted class:", outputs.logits[0].argmax().item())
print("Batch loss:", outputs.loss.item())

Logits shape: torch.Size([8, 5])
First example logits: tensor([ 0.4667,  0.5199,  0.3057, -0.3776,  0.0597], device='cuda:0')
Predicted class: 1
Batch loss: 1.5623152256011963


### 2.8 Configure AdamW and a learning-rate scheduler

**Explanation:** AdamW updates trainable parameters to reduce the loss. A small learning rate such as `2e-5` is commonly used when fine-tuning BERT because its pretrained weights already encode useful information.

The linear scheduler gradually reduces the learning rate during training. This example uses a warm-up period followed by a linear decay.

In [ ]:
EPOCHS = 2
LEARNING_RATE = 2e-5

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

total_training_steps = len(train_loader) * EPOCHS
warmup_steps = int(0.1 * total_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps,
)

print("Total optimizer steps:", total_training_steps)
print("Warm-up steps:", warmup_steps)

Total optimizer steps: 2000
Warm-up steps: 200


### 2.9 Write the training loop

**Explanation:** Every training batch follows the same learning cycle:

1. Move tensors to the selected device.
2. Clear old gradients with `zero_grad()`.
3. Run BERT's forward pass and calculate loss.
4. Call `backward()` to calculate gradients.
5. Call `optimizer.step()` to update parameters.
6. Call `scheduler.step()` to update the learning rate.

`model.train()` enables training behaviour. Gradient clipping helps prevent unusually large gradients from destabilising updates.

In [ ]:
def train_model(model, train_loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0.0

    for step, batch in enumerate(train_loader, start=1):
        batch = {key: value.to(device) for key, value in batch.items()}

        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        if step % 100 == 0:
            print(f"Step {step}/{len(train_loader)} | loss: {loss.item():.4f}")

    return total_loss / max(len(train_loader), 1)

### 2.10 Evaluate the classifier

**Explanation:** Evaluation measures performance on examples not used for gradient updates. `model.eval()` switches to evaluation behaviour, while `torch.no_grad()` avoids storing gradients and saves memory.

Accuracy is the fraction of predictions that exactly match the labels. Loss gives another view of how well the model fits the examples.

In [ ]:
def evaluate(model, data_loader, device):
    model.eval()
    total_loss = 0.0
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in data_loader:
            batch = {key: value.to(device) for key, value in batch.items()}
            outputs = model(**batch)

            total_loss += outputs.loss.item()
            predictions = outputs.logits.argmax(dim=-1)

            all_predictions.extend(predictions.cpu().tolist())
            all_labels.extend(batch["labels"].cpu().tolist())

    accuracy = np.mean(np.array(all_predictions) == np.array(all_labels))
    average_loss = total_loss / max(len(data_loader), 1)
    return average_loss, accuracy, all_predictions, all_labels

### 2.11 Run fine-tuning

**Explanation:** This is the point where the pretrained BERT parameters are updated using Yelp examples. The model is not being trained from random initialization; it is being adapted to a new task.

Training time depends on hardware, dataset size, sequence length, and batch size. On CPU, start with smaller limits and one epoch.

In [ ]:
history = []

for epoch in range(EPOCHS):
    train_loss = train_model(model, train_loader, optimizer, scheduler, device)
    test_loss, test_accuracy, _, _ = evaluate(model, test_loader, device)

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "test_loss": test_loss,
        "test_accuracy": test_accuracy,
    })

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"train loss: {train_loss:.4f} | "
        f"test loss: {test_loss:.4f} | "
        f"test accuracy: {test_accuracy:.4f}"
    )

Step 100/1000 | loss: 1.6696
Step 200/1000 | loss: 1.1402
Step 300/1000 | loss: 1.0166
Step 400/1000 | loss: 1.2201
Step 500/1000 | loss: 1.0977
Step 600/1000 | loss: 0.9499
Step 700/1000 | loss: 0.9620
Step 800/1000 | loss: 1.0474
Step 900/1000 | loss: 1.0637
Step 1000/1000 | loss: 0.8825
Epoch 1/2 | train loss: 1.1761 | test loss: 0.9889 | test accuracy: 0.5690
Step 100/1000 | loss: 0.5041
Step 200/1000 | loss: 0.4524
Step 300/1000 | loss: 0.7847
Step 400/1000 | loss: 0.9598
Step 500/1000 | loss: 0.6986
Step 600/1000 | loss: 1.0818
Step 700/1000 | loss: 0.6598
Step 800/1000 | loss: 0.6081
Step 900/1000 | loss: 0.5120
Step 1000/1000 | loss: 0.3820
Epoch 2/2 | train loss: 0.8231 | test loss: 0.9727 | test accuracy: 0.5815


### 2.12 Predict a rating for a new review

**Explanation:** Tokenize a new review exactly as during training, pass it through the fine-tuned model, and take the index of the largest logit. Yelp labels are zero-based: class `0` represents one star and class `4` represents five stars.

In [ ]:
def predict_yelp_rating(review):
    model.eval()
    inputs = tokenizer(
        review,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits
        probabilities = torch.softmax(logits, dim=-1)[0]

    predicted_class = int(probabilities.argmax().item())
    return {
        "predicted_label": predicted_class,
        "predicted_stars": predicted_class + 1,
        "probabilities": probabilities.cpu().numpy(),
    }

predict_yelp_rating("The food was excellent and the staff were very friendly.")

{'predicted_label': 3,
 'predicted_stars': 4,
 'probabilities': array([0.00183207, 0.00602748, 0.04130938, 0.59278715, 0.3580439 ],
       dtype=float32)}

### 2.13 Save the fine-tuned classifier

**Explanation:** `save_pretrained()` saves the model weights and configuration. Saving the tokenizer alongside the model ensures that future input text is converted into the same token IDs.

The output folder can later be loaded with `from_pretrained()`.

In [ ]:
CLASSIFIER_DIR = "bert-yelp-classifier"
model.save_pretrained(CLASSIFIER_DIR)
tokenizer.save_pretrained(CLASSIFIER_DIR)
print("Saved to:", CLASSIFIER_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: bert-yelp-classifier


## 3. Practical B — Masked Language Modeling on IMDB

### 3.1 Classification versus MLM

**Yelp classification:** Given a complete review, predict one of five labels.

**Masked Language Modeling:** Given a sentence with one or more hidden tokens, predict the original token(s).

Example:

`This is a [MASK] movie.`

The model learns to predict the missing token using the surrounding context. BERT is bidirectional: it can use both left and right context.

### 3.2 Load IMDB and a BERT masked-language model

**Explanation:** The IMDB dataset contains movie reviews and sentiment labels. For MLM, we use the review text and do not use the sentiment label: the objective is to reconstruct masked tokens, not to classify positive versus negative sentiment.

`AutoModelForMaskedLM` loads BERT with a vocabulary prediction head rather than a five-class sentiment head.

In [ ]:
imdb = load_dataset("stanfordnlp/imdb")

MLM_CHECKPOINT = "bert-base-uncased"
mlm_tokenizer = AutoTokenizer.from_pretrained(MLM_CHECKPOINT)
mlm_model = AutoModelForMaskedLM.from_pretrained(MLM_CHECKPOINT)

print(imdb)
print("Example:", imdb["train"][0]["text"][:400])

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

... (3 outputs left out)

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
Example: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student name


### 3.3 Select a manageable training subset

**Explanation:** IMDB contains many long reviews. Tokenizing and training on the complete dataset can be slow and memory-intensive. Use a subset first to verify that the workflow runs; increase the limits when you have sufficient compute.

In [ ]:
MLM_TRAIN_LIMIT = 5000
MLM_EVAL_LIMIT = 1000
MLM_MAX_LENGTH = 128

mlm_train_text = imdb["train"].shuffle(seed=SEED).select(
    range(min(MLM_TRAIN_LIMIT, len(imdb["train"])))
)
mlm_eval_text = imdb["test"].shuffle(seed=SEED).select(
    range(min(MLM_EVAL_LIMIT, len(imdb["test"])))
)

### 3.4 Tokenize the reviews

**Explanation:** We tokenize each review and truncate it to `MLM_MAX_LENGTH`. Unlike the Yelp section, we do not need sentiment labels. The data collator will create the masked inputs and token-level training labels dynamically.

In [ ]:
def tokenize_imdb(batch):
    return mlm_tokenizer(
        batch["text"],
        truncation=True,
        max_length=MLM_MAX_LENGTH,
    )

mlm_train = mlm_train_text.map(
    tokenize_imdb,
    batched=True,
    remove_columns=mlm_train_text.column_names,
)
mlm_eval = mlm_eval_text.map(
    tokenize_imdb,
    batched=True,
    remove_columns=mlm_eval_text.column_names,
)

print(mlm_train[0].keys())

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

dict_keys(['input_ids', 'token_type_ids', 'attention_mask'])


### 3.5 Create the MLM data collator

**Explanation:** `DataCollatorForLanguageModeling` pads examples in a batch and randomly selects tokens to mask. With `mlm_probability=0.15`, approximately 15% of eligible tokens are selected as prediction targets.

The collator creates labels in which non-target positions are set to `-100`. PyTorch's cross-entropy loss ignores those positions, so the model is trained to predict only the selected tokens.

In [ ]:
mlm_collator = DataCollatorForLanguageModeling(
    tokenizer=mlm_tokenizer,
    mlm=True,
    mlm_probability=0.15,
)

# Inspect one collated batch to see the generated input and labels.
preview_features = [mlm_train[i] for i in range(2)]
preview_batch = mlm_collator(preview_features)

print("Input shape:", preview_batch["input_ids"].shape)
print("Labels shape:", preview_batch["labels"].shape)
print("Number of masked targets:", (preview_batch["labels"] != -100).sum().item())

Input shape: torch.Size([2, 128])
Labels shape: torch.Size([2, 128])
Number of masked targets: 36


### 3.6 Decode an example of masking

**Explanation:** This inspection helps make the MLM objective visible. In `input_ids`, some original tokens have been replaced with `[MASK]` (or modified according to BERT's masking procedure). In `labels`, only selected original token IDs are retained; other positions are `-100`.

Because masking is random, rerunning this cell can produce different masked positions.

In [ ]:
input_tokens = mlm_tokenizer.convert_ids_to_tokens(preview_batch["input_ids"][0].tolist())
target_ids = preview_batch["labels"][0].tolist()
target_tokens = [
    mlm_tokenizer.convert_ids_to_tokens([token_id])[0] if token_id != -100 else "·"
    for token_id in target_ids
]

print("Model input:")
print(input_tokens)
print("\nPrediction targets (· means ignored):")
print(target_tokens)

Model input:
['[CLS]', 'there', 'is', 'no', 'relation', 'at', 'all', 'between', 'fort', '##ier', 'and', 'profile', '##r', 'but', '[MASK]', '[MASK]', '[MASK]', 'both', 'are', 'police', 'series', 'about', 'violent', '[MASK]', '.', '[MASK]', '##r', 'looks', 'crisp', '##y', ',', 'fort', '##ier', 'looks', 'classic', '.', 'profile', '##r', 'plots', 'are', '[MASK]', 'simple', '[MASK]', 'fort', '##ier', '[MASK]', 's', '##kova', 'are', '[MASK]', 'more', 'complicated', '.', '.', '.', 'fort', '##ier', 'looks', 'more', 'like', 'prime', '[MASK]', '[MASK]', 'if', 'we', 'have', 'to', 'spot', 'similarities', '.', '.', '.', 'the', 'main', 'character', 'is', 'weak', 'and', 'weird', '##o', ',', 'but', 'have', '"', 'clair', '##vo', '##yan', '[MASK]', '"', '.', 'people', 'like', 'to', 'compare', ',', 'to', 'judge', ',', 'to', 'evaluate', '.', 'how', 'about', 'just', 'enjoying', '[MASK]', 'funny', 'thing', 'too', ',', 'people', 'writing', 'fort', '##ier', 'looks', 'american', 'but', ',', 'on', 'the', 'other

### 3.7 Configure Hugging Face Trainer

**Explanation:** `Trainer` handles the training loop, optimizer setup, batching, evaluation scheduling, and checkpointing. This is a higher-level alternative to writing every PyTorch training step manually.

The model is still a PyTorch model, and the training objective is still MLM. The trainer simply manages much of the training machinery.

In [ ]:
training_args = TrainingArguments(
    output_dir="bert-imdb-mlm",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED,
)

mlm_trainer = Trainer(
    model=mlm_model,
    args=training_args,
    train_dataset=mlm_train,
    eval_dataset=mlm_eval,
    data_collator=mlm_collator,
)

### 3.8 Fine-tune the MLM model

**Explanation:** During training, the collator masks tokens, BERT predicts vocabulary scores at token positions, and the loss compares predictions with the original IDs at masked positions. Backpropagation updates the model weights.

This is domain-adaptive MLM fine-tuning: the model adapts its language representations to movie-review text. It is not training a sentiment classifier.

In [ ]:
mlm_train_result = mlm_trainer.train()
print(mlm_train_result)

Epoch,Training Loss,Validation Loss
1,2.227994,2.045762


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=625, training_loss=2.2545322265625, metrics={'train_runtime': 91.2889, 'train_samples_per_second': 54.771, 'train_steps_per_second': 6.846, 'total_flos': 329006016000000.0, 'train_loss': 2.2545322265625, 'epoch': 1.0})


### 3.9 Evaluate and save the MLM model

**Explanation:** Evaluation reports the MLM loss on the held-out review subset. A lower loss generally means the model predicts the masked tokens better on this evaluation data.

Save both model and tokenizer so the trained checkpoint can be reused.

In [ ]:
mlm_metrics = mlm_trainer.evaluate()
print(mlm_metrics)

MLM_OUTPUT_DIR = "bert-imdb-mlm-final"
mlm_trainer.save_model(MLM_OUTPUT_DIR)
mlm_tokenizer.save_pretrained(MLM_OUTPUT_DIR)
print("Saved to:", MLM_OUTPUT_DIR)

Training Loss,Validation Loss,Epoch
2.227994,2.030321,1


{'eval_loss': 2.0303211212158203}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: bert-imdb-mlm-final


### 3.10 Predict a missing token with a fill-mask pipeline

**Explanation:** A `fill-mask` pipeline takes a sentence containing the model's mask token and returns likely replacements. Each result includes `token_str` (the predicted token) and `score` (its model score).

Use the tokenizer's actual mask token rather than typing a mask token from another model family.

In [ ]:
fill_mask = pipeline(
    "fill-mask",
    model=MLM_OUTPUT_DIR,
    tokenizer=MLM_OUTPUT_DIR,
    device=0 if torch.cuda.is_available() else -1,
)

masked_sentence = f"This is a {mlm_tokenizer.mask_token} movie!"
predictions = fill_mask(masked_sentence, top_k=5)

for item in predictions:
    print(f"Token: {item['token_str']:<15} Score: {item['score']:.4f}")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Token: great           Score: 0.3831
Token: wonderful       Score: 0.1454
Token: good            Score: 0.0736
Token: fantastic       Score: 0.0629
Token: terrible        Score: 0.0314


## 4. PyTorch loop versus Hugging Face Trainer

| Topic | Manual PyTorch loop | Hugging Face `Trainer` |
|---|---|---|
| Training loop | You write it | Trainer manages it |
| Optimizer | You configure and call it | Configured through training arguments |
| Batching | You create DataLoaders | Trainer creates batches |
| Loss/backpropagation | You handle the steps | Trainer handles the steps |
| Control | More direct control | Higher-level workflow |
| Best learning value | Understand every training step | Run standard experiments efficiently |

**Remember:** Hugging Face Transformers and PyTorch work together. Transformers supplies model implementations and utilities; PyTorch provides the underlying tensor and autograd framework.

## 5. Important concepts to remember

| Concept | Meaning |
|---|---|
| `load_dataset()` | Loads a dataset from the Hugging Face Hub |
| `AutoTokenizer` | Converts text into token IDs and masks |
| `input_ids` | Numerical token representation |
| `attention_mask` | Distinguishes real tokens from padding |
| Padding | Makes examples in a batch the same length |
| Truncation | Limits sequences to a maximum length |
| `num_labels=5` | Five output classes for Yelp |
| `AutoModelForSequenceClassification` | Model with a classification head |
| `AdamW` | Optimizer that updates trainable parameters |
| `loss.backward()` | Calculates gradients |
| `optimizer.step()` | Updates parameters |
| `AutoModelForMaskedLM` | Model with a masked-token prediction head |
| `DataCollatorForLanguageModeling` | Creates masked-token examples |
| `Trainer` | Manages much of the training workflow |
| `fill-mask` | Predicts masked tokens |

### The most important difference

- **Sequence classification:** one prediction for the whole input (for example, a Yelp rating).
- **Masked language modeling:** token predictions at masked positions (for example, filling `[MASK]` in a sentence).
- **Supervised fine-tuning (SFT):** commonly used to train generative models on prompt/response examples. It is not the same objective as BERT MLM.

## 6. Knowledge check

Try to answer before looking at the answers.

1. What does `num_labels=5` mean?
2. What does an attention mask identify?
3. Why do we use padding and truncation?
4. What does `loss.backward()` calculate?
5. What does the optimizer do after gradients are calculated?
6. In MLM, why are most label positions set to `-100`?
7. What is the difference between Yelp classification and IMDB MLM?
8. Does using Hugging Face `Trainer` mean the model is no longer a PyTorch model?

### Answers

1. The classification head has five output classes.
2. It distinguishes actual tokens from padding tokens.
3. They make batch inputs compatible and keep sequence length manageable.
4. Gradients of the loss with respect to trainable parameters.
5. It updates model parameters using the gradients.
6. Those positions are ignored by the loss; only selected masked-token targets contribute.
7. Yelp predicts a review rating; MLM predicts missing tokens in review text.
8. No. Trainer manages the workflow around a PyTorch model.

## 7. Suggested experiments

1. Change `MAX_LENGTH` from 128 to 64 and compare speed and accuracy.
2. Increase the Yelp subset gradually and compare results.
3. Try batch sizes of 4 and 8, depending on available memory.
4. Compare one versus two Yelp fine-tuning epochs.
5. Change `mlm_probability` from 0.15 to 0.20 and observe the number of masked targets.
6. Inspect several fill-mask predictions and consider whether the sentence context supports them.

**Good practice:** Record the dataset subset, random seed, model checkpoint, learning rate, batch size, sequence length, and epoch count for every experiment. This makes results reproducible.